Building & Fine-Tuning a Domain-Specific LLM

Model Used - HuggingFaceTB/SmolLM2-360M

# Domain Data Sources

* **Primary Source:** [help.ubuntu.com](https://help.ubuntu.com) *(Ubuntu Official Help & Manuals)*
* **Dataset Scale:** 1,140 pages covering official desktop and server guides.
* **Scope & Usage:** Acts as the base dataset for all data processing, fine-tuning, and model evaluation tasks.

---

### Included Ubuntu Releases

| Release Version | Code Name | Support Status |
| :--- | :--- | :--- |
| **Ubuntu 26.04 LTS** | *Resolute Raccoon* | Active LTS |
| **Ubuntu 24.04 LTS** | *Noble Numbat* | Active LTS |
| **Ubuntu 22.04 LTS** | *Jammy Jellyfish* | Active LTS |

# Step 1 — Data Collection, Extraction & Cleaning

In [ ]:
# one time process then restart notebook and run all cells
!pip install pypdf langdetect
!pip install trl bitsandbytes

In [ ]:
from pypdf import PdfReader, PdfWriter
!mkdir raw_data
def extract_ranges(input_pdf: str, ranges: dict):
    """Extract specified page ranges from a PDF and save them as individual document subsets."""
    reader = PdfReader(input_pdf)
    total_pages = len(reader.pages)

    for filename, (start, end) in ranges.items():
        writer = PdfWriter()
        actual_end = min(end, total_pages)
        for page_num in range(start - 1, actual_end):
            writer.add_page(reader.pages[page_num])

        with open(filename, "wb") as out:
            writer.write(out)
        print(f"Created: {filename} (Pages {start}–{actual_end})")


pdf_splits = {
    "./raw_data/01_ubuntu_tutorial.pdf": (7, 75),
    "./raw_data/02_ubuntu_howto_part1.pdf": (75, 351),
    "./raw_data/03_ubuntu_howto_part2.pdf": (351, 792),
    "./raw_data/04_ubuntu_reference.pdf": (792, 866),
    "./raw_data/05_ubuntu_explanations.pdf": (866, 1113),
}

# Execute target document extraction
extract_ranges("./canonical-ubuntu-server-documentation-readthedocs-hosted-com-en-latest.pdf", pdf_splits)

# Step 2 — Tokenization & Packed Dataset

In [ ]:
import os
import re
from pathlib import Path
from pypdf import PdfReader
from langdetect import detect, DetectorFactory

!mkdir domain_corpus
# Enforce deterministic language detection
DetectorFactory.seed = 0

INPUT_DIR = "./raw_data"        # Source folder for extracted PDFs
OUTPUT_DIR = "./domain_corpus"  # Destination folder for cleaned txt files
MIN_PAGE_CHAR_LENGTH = 150      # Threshold to filter blank or non-substantive pages

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def clean_text_content(text: str) -> str:
    """Performs structural and whitespace normalization on raw text."""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

# Tracking structural metrics across dataset
total_pages_extracted = 0
pages_after_length = 0
pages_after_dedup = 0
pages_after_lang = 0

seen_page_hashes = set()
pdf_files = sorted(list(Path(INPUT_DIR).glob("*.pdf")))

print(f"--- Processing {len(pdf_files)} PDFs Page-by-Page ---")

for pdf_path in pdf_files:
    try:
        reader = PdfReader(pdf_path)
        pdf_pages_retained = []

        for i, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text()
            if not page_text:
                continue

            cleaned_page = clean_text_content(page_text)
            total_pages_extracted += 1

            # Apply character length threshold filter
            if len(cleaned_page) < MIN_PAGE_CHAR_LENGTH:
                continue
            pages_after_length += 1

            # Apply page-level content deduplication
            page_hash = hash(" ".join(cleaned_page.split()))
            if page_hash in seen_page_hashes:
                continue
            seen_page_hashes.add(page_hash)
            pages_after_dedup += 1

            # Verify document language
            try:
                if detect(cleaned_page[:500]) != "en":
                    continue
            except Exception:
                pass # Maintain original sample if evaluation fails
            pages_after_lang += 1

            pdf_pages_retained.append(cleaned_page)

        # Consolidate and export cleaned text segments
        if pdf_pages_retained:
            combined_document_text = "\n\n".join(pdf_pages_retained)
            out_filename = f"{pdf_path.stem}_cleaned.txt"
            out_path = Path(OUTPUT_DIR) / out_filename

            with open(out_path, "w", encoding="utf-8") as f:
                f.write(combined_document_text)
            print(f"Saved: {out_filename} ({len(pdf_pages_retained)} unique pages retained)")

    except Exception as e:
        print(f"Error processing {pdf_path.name}: {e}")

# Compute pipeline impact statistics
drop_length = total_pages_extracted - pages_after_length
drop_dedup = pages_after_length - pages_after_dedup
drop_lang = pages_after_dedup - pages_after_lang

impact_dict = {
    "Length Filter": drop_length,
    "Deduplication": drop_dedup,
    "Language Filter": drop_lang
}

greatest_step = max(impact_dict, key=impact_dict.get)

print("\n================ PAGE-LEVEL CLEANING REPORT ================")
print(f"Total Raw Pages Extracted  : {total_pages_extracted}")
print(f"After Page Length Filter   : {pages_after_length}  (Removed: {drop_length} junk/short pages)")
print(f"After Page Deduplication  : {pages_after_dedup}  (Removed: {drop_dedup} duplicate pages)")
print(f"After Page Language Filter : {pages_after_lang}  (Removed: {drop_lang} non-English pages)")
print("---------------------------------------------------------")
print(f"Final Retained Pages       : {pages_after_lang}")
print(f"Step with Greatest Impact  : {greatest_step} (Removed {impact_dict[greatest_step]} pages)")
print(f"Output Saved To            : {OUTPUT_DIR} (1 merged .txt per PDF)")
print("=========================================================\n")

In [ ]:
import glob
import os
import pandas as pd
from transformers import AutoTokenizer

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
CORPUS_DIR = "./domain_corpus"
PARQUET_OUTPUT = "packed_dataset.parquet"
CONTEXT_LENGTH = 512  # Sequence packing block size

# ==============================================================================
# 2. LOAD TOKENIZER (PRESERVE VOCABULARY ALIGNMENT)
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Ensure BOS/EOS token IDs are resolved correctly
bos_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.eos_token_id
eos_id = tokenizer.eos_token_id

print(f"Loaded Tokenizer : {MODEL_ID}")
print(f"BOS Token ID     : {bos_id} ({tokenizer.decode([bos_id])})")
print(f"EOS Token ID     : {eos_id} ({tokenizer.decode([eos_id])})")

# ==============================================================================
# 3. TOKENIZATION WITH DOCUMENT-LEVEL BOUNDARIES
# ==============================================================================
txt_files = sorted(glob.glob(os.path.join(CORPUS_DIR, "*.txt")))
if not txt_files:
    raise FileNotFoundError(f"No .txt files found in '{CORPUS_DIR}'. Please run Step 1 first.")

doc_token_counts = []
flat_token_stream = []

print(f"\n--- Tokenizing {len(txt_files)} Documents ---")

for filepath in txt_files:
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read().strip()

    # Encode raw text without automatic special tokens
    raw_tokens = tokenizer.encode(text, add_special_tokens=False)

    # Wrap DOCUMENT level: [BOS] + raw_tokens + [EOS]
    wrapped_tokens = [bos_id] + raw_tokens + [eos_id]

    doc_length = len(wrapped_tokens)
    doc_token_counts.append(doc_length)
    flat_token_stream.extend(wrapped_tokens)
    print(f"Processed '{os.path.basename(filepath)}': {doc_length:,} tokens")

# ==============================================================================
# 4. SEQUENCE PACKING (512-TOKEN BLOCKS)
# ==============================================================================
total_tokens = len(flat_token_stream)
avg_doc_length = sum(doc_token_counts) / len(doc_token_counts) if doc_token_counts else 0

packed_chunks = []
for i in range(0, total_tokens, CONTEXT_LENGTH):
    chunk = flat_token_stream[i : i + CONTEXT_LENGTH]
    # Keep only complete 512-token blocks for zero-padding GPU efficiency
    if len(chunk) == CONTEXT_LENGTH:
        packed_chunks.append(chunk)

num_packed_sequences = len(packed_chunks)

# ==============================================================================
# 5. SAVE PACKED DATASET & REPORT METRICS
# ==============================================================================
df_packed = pd.DataFrame({"input_ids": packed_chunks})
df_packed.to_parquet(PARQUET_OUTPUT, index=False)

print("\n================ STEP 2 REPORT ================")
print(f"Model Tokenizer             : {MODEL_ID}")
print(f"Total Text Documents        : {len(txt_files)}")
print(f"Total Token Count           : {total_tokens:,}")
print(f"Average Doc Length (Tokens) : {avg_doc_length:,.2f}")
print(f"Context Length Block Size   : {CONTEXT_LENGTH}")
print(f"Total Packed Sequences      : {num_packed_sequences:,}")
print(f"Saved Packed Dataset To     : {PARQUET_OUTPUT}")
print("===============================================\n")

# Step 3 — Model Loading & Architecture Inspection

In [ ]:
import torch
from transformers import AutoModelForCausalLM
MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
CORPUS_DIR = "./domain_corpus"
PARQUET_OUTPUT = "packed_dataset.parquet"
CONTEXT_LENGTH = 512  # Sequence packing block size
# ==============================================================================
# 1. SETUP HARDWARE PRECISION & LOAD MODEL
# ==============================================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# Reuses MODEL_ID from previous cell
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None
)

# Enable Gradient Checkpointing for VRAM efficiency during training
model.gradient_checkpointing_enable()

# ==============================================================================
# 2. ARCHITECTURE AUDIT & PARAMETER COUNT
# ==============================================================================
config = model.config
total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)

print("================ STEP 3: MODEL ARCHITECTURE AUDIT ================")
print(f"Loaded Precision           : {dtype}")
print(f"Total Trainable Parameters : {total_trainable:,}")
print(f"Decoder Layers             : {config.num_hidden_layers}")
print(f"Attention Heads            : {config.num_attention_heads}")
print(f"Hidden Dimension           : {config.hidden_size}")
print(f"Head Dimension             : {head_dim}")
print(f"LM Head Output Dimension   : {model.lm_head.out_features}")
print(f"Config Vocab Size          : {config.vocab_size}")
print(f"Matches Vocab Size?        : {model.lm_head.out_features == config.vocab_size}")
print("===================================================================\n")



In [ ]:
import torch

domain_prompts = [
    "How do I configure a static IP address on Ubuntu Server using Netplan?",
    "How do I create and enable a custom systemd service on Ubuntu?",
    "How do I automatically mount a secondary hard drive on boot in Ubuntu using its UUID in /etc/fstab?"
]


baseline_results = []
model.eval()

for i, prompt in enumerate(domain_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        out_tokens = model.generate(
            **inputs,
            max_new_tokens=256,  # Generates complete responses without truncation
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    out_text = tokenizer.decode(out_tokens[0], skip_special_tokens=True)
    baseline_results.append((prompt, out_text))




In [ ]:
import textwrap

print("Prompt 1:")
print(textwrap.fill(baseline_results[0][0], width=100))

print("\nResponse:")
print(textwrap.fill(baseline_results[0][1], width=100))

In [ ]:


print("Prompt 2:")
print(textwrap.fill(baseline_results[1][0], width=100))

print("\nResponse:")
print(textwrap.fill(baseline_results[1][1], width=100))

In [ ]:
print("Prompt 3:")
print(textwrap.fill(baseline_results[2][0], width=100))

print("\nResponse:")
print(textwrap.fill(baseline_results[2][1], width=100))

# Step 4 — CPT Training Loop & Loss Analysis

In [ ]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import (
    Trainer,
    TrainingArguments,
    TrainerCallback,
    EarlyStoppingCallback
)

# ==============================================================================
# 1. OPTIMIZED DATA LOADING & TRAIN/VAL SPLIT
# ==============================================================================
parquet_file = "packed_dataset.parquet"
df = pd.read_parquet(parquet_file)

full_dataset = Dataset.from_pandas(df[["input_ids"]])

def prepare_cpt_inputs(example):
    tokens = example["input_ids"]
    return {
        "input_ids": tokens,
        "labels": tokens,  # Causal LM auto-shifts loss internally
        "attention_mask": [1] * len(tokens)
    }

# Reduced num_proc to 2 to match system CPU limits
full_dataset = full_dataset.map(prepare_cpt_inputs, batched=False, num_proc=2)

split_data = full_dataset.train_test_split(test_size=0.10, seed=42)
train_dataset = split_data["train"]
eval_dataset = split_data["test"]

print(f"Loaded datasets -> Train samples: {len(train_dataset):,} | Validation samples: {len(eval_dataset):,}")

# ==============================================================================
# 2. CUSTOM LOSS LOGGING CALLBACK
# ==============================================================================
class LossLoggingCallback(TrainerCallback):
    def __init__(self):
        self.steps = []
        self.losses = []
        self.eval_steps = []
        self.eval_losses = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if "loss" in logs:
                self.steps.append(state.global_step)
                self.losses.append(logs["loss"])
            if "eval_loss" in logs:
                self.eval_steps.append(state.global_step)
                self.eval_losses.append(logs["eval_loss"])

loss_callback = LossLoggingCallback()

# ==============================================================================
# 3. TRAINING ARGUMENTS & TRAINER SETUP (FIXED CHECKPOINTING)
# ==============================================================================
OUTPUT_DIR = "./cpt_smollm2_ubuntu"

model.config.use_cache = False  # Critical during training to allow gradient checkpointing

total_train_steps = (len(train_dataset) // (8 * 16)) * 5
calculated_warmup_steps = max(1, int(0.05 * total_train_steps))  # 5% warmup

eval_strategy_key = "eval_strategy" if hasattr(TrainingArguments, "eval_strategy") else "evaluation_strategy"

# Updated config for small dataset CPT (~1,112 samples)
training_args_dict = {
    "output_dir": OUTPUT_DIR,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 5e-5,                 # Reduced LR to avoid rapid overfitting
    "lr_scheduler_type": "cosine",
    "warmup_steps": 10,
    "weight_decay": 0.01,
    "num_train_epochs": 3,
    "eval_steps": 10,                      # Evaluate every 10 steps (~every 160 samples)
    "save_steps": 10,
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "dataloader_num_workers": 2,
    "bf16": torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    "fp16": not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": True},
    "logging_steps": 5,
    "report_to": "none"
}

training_args_dict[eval_strategy_key] = "steps"
training_args = TrainingArguments(**training_args_dict)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    callbacks=[
        loss_callback,
        EarlyStoppingCallback(early_stopping_patience=3)
    ]
)

print("\n--- STARTING CONTINUAL PRE-TRAINING (CPT) ---")
trainer.train()
print("--- CPT COMPLETED SUCCESSFULLY ---")




In [ ]:
# ==============================================================================
# 4. PLOT LOSS CURVES (Train vs Validation)
# ==============================================================================
if loss_callback.losses:
    plt.figure(figsize=(10, 5))
    plt.plot(loss_callback.steps, loss_callback.losses, label="Train Loss", color='#1f77b4', linewidth=2)
    if loss_callback.eval_losses:
        plt.plot(loss_callback.eval_steps, loss_callback.eval_losses, label="Validation Loss", color='#ff7f0e', linewidth=2, linestyle='--')

    plt.title("CPT Loss Curve (Train vs Validation)")
    plt.xlabel("Global Step")
    plt.ylabel("Cross-Entropy Loss")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig("cpt_loss_curve.png", dpi=300)
    plt.show()

In [ ]:
from transformers import AutoTokenizer
# ==============================================================================
# 5. POST-CPT EVALUATION
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("\n================ QUALITATIVE COMPARISON (POST-CPT) ================")
model.eval()
device = next(model.parameters()).device
post_cpt_results = []

for i, prompt in enumerate(domain_prompts, 1):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        out_tokens = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    gen_text = tokenizer.decode(out_tokens[0], skip_special_tokens=True)
    post_cpt_results.append((prompt, gen_text))


# ==============================================================================
# 6. SAVE MODEL AND TOKENIZER
# ==============================================================================
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nSaved optimal CPT model & tokenizer to: '{OUTPUT_DIR}'")

In [ ]:

print("Prompt 1:")
print(textwrap.fill(post_cpt_results[0][0], width=100))

print("\nResponse:")
print(textwrap.fill(post_cpt_results[0][1], width=100))

In [ ]:

print("Prompt 2:")
print(textwrap.fill(post_cpt_results[1][0], width=100))

print("\nResponse:")
print(textwrap.fill(post_cpt_results[1][1], width=100))

In [ ]:

print("Prompt 3:")
print(textwrap.fill(post_cpt_results[2][0], width=100))

print("\nResponse:")
print(textwrap.fill(post_cpt_results[2][1], width=100))

# Step 5 — Evaluation: Perplexity & Catastrophic Forgetting

## 5A — Domain Perplexity

In [ ]:
import math
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

# ==============================================================================
# 1. RECREATE EVAL_DATASET FROM PARQUET FILE
# ==============================================================================
parquet_file = "packed_dataset.parquet"
df = pd.read_parquet(parquet_file)

full_dataset = Dataset.from_pandas(df[["input_ids"]])

def prepare_cpt_inputs(example):
    tokens = example["input_ids"]
    return {
        "input_ids": tokens,
        "labels": tokens,  # Causal LM auto-shifts loss internally
        "attention_mask": [1] * len(tokens)
    }

# Process dataset
full_dataset = full_dataset.map(prepare_cpt_inputs, batched=False, num_proc=2)

# Same seed and test_size ensure the exact same 10% split as training
split_data = full_dataset.train_test_split(test_size=0.10, seed=42)
eval_dataset = split_data["test"]

print(f"Successfully loaded evaluation set: {len(eval_dataset):,} samples")

# ==============================================================================
# 2. EVALUATION CONFIGURATION
# ==============================================================================
eval_args = TrainingArguments(
    output_dir="./eval_tmp",
    per_device_eval_batch_size=4,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
    report_to="none"
)

# ==============================================================================
# 3. EVALUATE BASE MODEL PPL
# ==============================================================================
BASE_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"

print("\n--- EVALUATING BASE MODEL PERPLEXITY ---")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)

base_trainer = Trainer(
    model=base_model,
    args=eval_args,
    eval_dataset=eval_dataset
)

base_metrics = base_trainer.evaluate()
base_loss = base_metrics["eval_loss"]
base_ppl = math.exp(base_loss)

# Memory cleanup before loading CPT model
del base_model
del base_trainer
torch.cuda.empty_cache()

# ==============================================================================
# 4. EVALUATE CPT MODEL PPL
# ==============================================================================
CPT_MODEL_PATH = "./cpt_smollm2_ubuntu"

print("\n--- EVALUATING CPT MODEL PERPLEXITY ---")
cpt_model = AutoModelForCausalLM.from_pretrained(CPT_MODEL_PATH)

cpt_trainer = Trainer(
    model=cpt_model,
    args=eval_args,
    eval_dataset=eval_dataset
)

cpt_metrics = cpt_trainer.evaluate()
cpt_loss = cpt_metrics["eval_loss"]
cpt_ppl = math.exp(cpt_loss)

# Memory cleanup
del cpt_model
del cpt_trainer
torch.cuda.empty_cache()

# ==============================================================================
# 5. STEP 5A REPORT
# ==============================================================================
ppl_reduction = ((base_ppl - cpt_ppl) / base_ppl) * 100

print("\n================ STEP 5A EVALUATION REPORT ================")
print(f"Base Model Perplexity (PPL) : {base_ppl:.2f}")
print(f"CPT Model Perplexity (PPL)  : {cpt_ppl:.2f}")
print(f"Perplexity Reduction (%)    : {ppl_reduction:.2f}%")
print("==========================================================")

## 5B — Catastrophic Forgetting Check

In [ ]:
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ==============================================================================
# 1. SETUP PROMPTS & TEST CASES
# ==============================================================================
prompts = [
    "The capital of France is",
    "Water boils at",
    "The speed of light is approximately"
]

# Expected key facts to verify accuracy programmatically
expected_keywords = [
    ["paris"],
    ["100", "212"],
    ["300", "299", "3x10^8", "3 x 10^8", "186,000", "300,000", "186000"]
]

# ==============================================================================
# 2. LOAD TOKENIZER AND MODELS
# ==============================================================================
BASE_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
CPT_MODEL_PATH = "./SmolLM2_CPT_Project/cpt_smollm2_ubuntu"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Base Model...")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME).to("cuda" if torch.cuda.is_available() else "cpu")

print("Loading CPT Model...")
cpt_model = AutoModelForCausalLM.from_pretrained(CPT_MODEL_PATH).to("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# 3. TEXT GENERATION HELPER
# ==============================================================================
def generate_text(model, prompt, max_new_tokens=25):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # Greedy decoding for deterministic evaluation
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id
        )
    # Decode only the newly generated tokens
    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# ==============================================================================
# 4. RUN COMPARISON & EVALUATION
# ==============================================================================
results = []

for prompt, keywords in zip(prompts, expected_keywords):
    base_out = generate_text(base_model, prompt)
    cpt_out = generate_text(cpt_model, prompt)

    # Check if the generated response retains factual core knowledge
    base_correct = any(kw in base_out.lower() for kw in keywords)
    cpt_correct = any(kw in cpt_out.lower() for kw in keywords)

    if cpt_correct or (not base_correct and not cpt_correct):
        verdict = "Retained"
    else:
        verdict = "Degraded"

    results.append({
        "Prompt": prompt,
        "Base Model Output": base_out,
        "CPT Model Output": cpt_out,
        "Verdict": verdict
    })

# Memory cleanup
del base_model
del cpt_model
torch.cuda.empty_cache()

# ==============================================================================
# 5. DISPLAY RESULTS TABLE (Step 5B)
# ==============================================================================
df_results = pd.DataFrame(results)


In [ ]:
import textwrap

print("\n================ STEP 5B: CATASTROPHIC FORGETTING REPORT ================\n")

pd.set_option('display.max_colwidth', 100)

for _, row in df_results.iterrows():
    print("-" * 100)
    for col in df_results.columns:
        value = str(row[col])
        print(f"{col}:")
        print(textwrap.fill(value, width=100))
    print()

# Step 5: CPT Performance Evaluation

## Perplexity Improvement

The Continued Pretraining (CPT) model was evaluated against the original Base Model on a held-out evaluation set of **118 samples**.

| Metric | Base Model | CPT Model | Improvement |
|---|---:|---:|---:|
| Validation Loss | 2.1536 | 2.0422 | ↓ 5.17% |
| Perplexity (PPL) | 8.62 | 7.71 | ↓ 10.54% |

### Key Finding

The CPT model achieved a **10.54% reduction in perplexity**, decreasing from **8.62 to 7.71**. This indicates that continued pretraining improved the model's ability to model the target Ubuntu/technical domain and better predict tokens from the evaluation corpus.

The corresponding validation loss also decreased from **2.1536 to 2.0422**, providing additional evidence of improved language modeling performance after CPT.

## Catastrophic Forgetting Evaluation

A small qualitative evaluation was performed using general-knowledge prompts to check whether CPT negatively affected the model's existing knowledge.

| Knowledge Area | Base Model | CPT Model | Verdict |
|---|---|---|---|
| Capital of France / USA | Correct | Correct | Retained |
| Boiling point of water | Correct | Correct | Retained |
| Speed of light | Correct | Correct | Retained |

### Conclusion

The evaluation shows that **CPT improved domain-specific performance without obvious catastrophic forgetting on the tested general-knowledge examples**.

Overall:

- **Perplexity improved:** 8.62 → 7.71
- **Perplexity reduction:** **10.54%**
- **Validation loss improved:** 2.1536 → 2.0422
- **General knowledge:** Retained on the qualitative test set

> **Result:** Continued pretraining successfully improved the model's target-domain language modeling performance while preserving the tested general knowledge.

# PART B — Instruction Fine-Tuning with QLoRA

# Part B1 — Instruction Dataset Creation

We used a local model llama3.1:8B on personal system to obtain the instruction dataset

Prompt used>

'''

You are an expert technical AI dataset creator.

Task: Read the domain text below and generate 5 distinct, high-quality instruction-response pairs derived strictly from the technical information in the text.

CRITICAL RULES:
1. USER PERSPECTIVE: The instruction (Q) MUST be framed as a genuine question or task from a system administrator or developer (e.g., "What are the supported architectures for Ubuntu Server?").
2. FORBIDDEN IN QUESTIONS & ANSWERS:
   - Do NOT include documentation metadata or external pointers (e.g., "refer to this page", "check page 75", "visit the website for tutorials").
   - Do NOT mention links, UI buttons ("Give feedback"), or book sections ("Reference section").
3. SELF-CONTAINED ANSWERS: Every response (A) MUST be direct, standalone, and informative using only facts from the text.
4. TECHNICAL SUBSTANCE: Focus on concrete commands, hardware requirements, architectures, and step-by-step procedures.

Domain Text:

{text_chunk}

'''

In [ ]:
import os
import glob
import json
import pandas as pd
from typing import List
from pydantic import BaseModel, Field
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

OUTPUT_DIR = "./instruction_dataset"
RAW_JSONL_PATH = os.path.join('./SmolLM2_CPT_Project/', "all_generated_pairs.jsonl")
if os.path.exists(RAW_JSONL_PATH):
    print(f"Data for Instruction tuning exists ")
else:
  # ==============================================================================
  # 1. DEFINE PYDANTIC SCHEMAS FOR STRUCTURED OUTPUT
  # ==============================================================================
  class InstructionPair(BaseModel):
      instruction: str = Field(
          description="A clear, realistic user prompt or question derived directly from the domain text."
      )
      response: str = Field(
          description="An accurate, comprehensive answer grounded strictly in the provided domain text."
      )

  class InstructionDataset(BaseModel):
      pairs: List[InstructionPair] = Field(
          description="A list of generated instruction-response pairs."
      )

  # ==============================================================================
  # 2. LOAD CHUNKS FROM A SINGLE FILE
  # ==============================================================================
  def load_text_chunks_from_file(file_path, chunk_size=1000):
      text_chunks = []

      if file_path.endswith(".parquet"):
          df = pd.read_parquet(file_path)
          text_column = "text" if "text" in df.columns else df.columns[0]
          raw_text = " ".join(df[text_column].dropna().astype(str).tolist())
      else:
          with open(file_path, "r", encoding="utf-8") as f:
              raw_text = f.read()

      words = raw_text.split()
      for i in range(0, len(words), chunk_size):
          chunk = " ".join(words[i:i + chunk_size])
          if chunk.strip():
              text_chunks.append(chunk)

      return text_chunks

  # ==============================================================================
  # 3. LOAD HUGGING FACE MODEL
  # ==============================================================================
  MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

  tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
  model = AutoModelForCausalLM.from_pretrained(
      MODEL_NAME,
      torch_dtype="auto",
      device_map="auto"
  )

  generator = pipeline(
      "text-generation",
      model=model,
      tokenizer=tokenizer
  )

  # ==============================================================================
  # 4. HUGGING FACE CHAT CALL
  # ==============================================================================
  def generate_structured_pairs(text_chunk, model=MODEL_NAME, temperature=0.2):
      messages = [
          {
              "role": "user",
              "content": (
                  f"""You are an expert technical AI dataset creator.

  Task: Read the domain text below and generate 5 distinct, high-quality instruction-response pairs derived strictly from the technical information in the text.

  CRITICAL RULES:
  1. USER PERSPECTIVE: The instruction (Q) MUST be framed as a genuine question or task from a system administrator or developer (e.g., "What are the supported architectures for Ubuntu Server?").
  2. FORBIDDEN IN QUESTIONS & ANSWERS:
    - Do NOT include documentation metadata or external pointers (e.g., "refer to this page", "check page 75", "visit the website for tutorials").
    - Do NOT mention links, UI buttons ("Give feedback"), or book sections ("Reference section").
  3. SELF-CONTAINED ANSWERS: Every response (A) MUST be direct, standalone, and informative using only facts from the text.
  4. TECHNICAL SUBSTANCE: Focus on concrete commands, hardware requirements, architectures, and step-by-step procedures.

  Domain Text:
  {text_chunk}"""
              )
          }
      ]

      try:
          response = generator(
              messages,
              max_new_tokens=2048,
              temperature=temperature,
              do_sample=True
          )

          content = response[0]["generated_text"][-1]["content"]

          parsed_data = InstructionDataset.model_validate_json(content)
          return [pair.model_dump() for pair in parsed_data.pairs]

      except Exception as e:
          print(f"Error during extraction: {e}")
          return []


  # OPTION B: Or automatically pick up all .txt / .parquet files in a folder
  CORPUS_DIR = "./domain_corpus"
  INPUT_FILES = glob.glob(os.path.join(CORPUS_DIR, "*.txt")) + glob.glob(os.path.join(CORPUS_DIR, "*.parquet"))

  OUTPUT_DIR = "./instruction_dataset"
  os.makedirs(OUTPUT_DIR, exist_ok=True)
  RAW_JSONL_PATH = os.path.join(OUTPUT_DIR, "all_generated_pairs.jsonl")

  newly_generated_pairs = []

  print(f"Found {len(INPUT_FILES)} input file(s) to process.\n")

  # Open file in append mode to log all pairs from all files consecutively
  with open(RAW_JSONL_PATH, "a", encoding="utf-8") as raw_file:
      for file_idx, file_path in enumerate(INPUT_FILES, 1):
          filename = os.path.basename(file_path)
          print(f"\n==================================================")
          print(f" Processing File {file_idx}/{len(INPUT_FILES)}: {filename}")
          print(f"==================================================")

          chunks = load_text_chunks_from_file(file_path, chunk_size=1000)
          print(f"Loaded {len(chunks)} chunks from {filename}.")

          for chunk_idx, chunk in enumerate(chunks, 1):
              print(f"\n--- [File {file_idx}/{len(INPUT_FILES)}] Chunk {chunk_idx}/{len(chunks)} ---")
              pairs = generate_structured_pairs(chunk)

              for p_idx, pair in enumerate(pairs, 1):
                  print(f"  [Pair {p_idx}] Q: {pair['instruction'][:70]}...")

                  raw_file.write(json.dumps(pair, ensure_ascii=False) + "\n")
                  raw_file.flush()

              newly_generated_pairs.extend(pairs)
              print(f"  -> Completed chunk {chunk_idx}. Appended {len(pairs)} pairs.")

  print(f"\nNew Pairs Written To Disk Across All Files: {len(newly_generated_pairs):,}")

  # ==============================================================================
  # 5. LOAD ALL CUMULATIVE PAIRS FROM DISK & SPLIT 80/20
  # ==============================================================================
  all_accumulated_pairs = []
  seen_instructions = set()
  duplicates_skipped = 0

  if os.path.exists(RAW_JSONL_PATH):
      with open(RAW_JSONL_PATH, "r", encoding="utf-8") as f:
          for line in f:
              if line.strip():
                  data = json.loads(line)
                  instruction_key = data["instruction"].strip().lower()

                  if instruction_key not in seen_instructions:
                      seen_instructions.add(instruction_key)
                      all_accumulated_pairs.append(data)
                  else:
                      duplicates_skipped += 1

  if not all_accumulated_pairs:
      raise ValueError("No instruction pairs were saved to disk!")

  hf_dataset = Dataset.from_list(all_accumulated_pairs)

  split_dataset = hf_dataset.train_test_split(test_size=0.20, seed=42)
  train_data = split_dataset["train"]
  eval_data = split_dataset["test"]

  train_path = os.path.join(OUTPUT_DIR, "train_instructions.jsonl")
  eval_path = os.path.join(OUTPUT_DIR, "eval_instructions.jsonl")

  train_data.to_json(train_path, orient="records", lines=True)
  eval_data.to_json(eval_path, orient="records", lines=True)

  # ==============================================================================
  # 6. FINAL METRICS REPORT
  # ==============================================================================
  print("\n================ PART B1 MULTI-FILE DATASET REPORT ================")
  print(f"Files Processed        : {len(INPUT_FILES)}")
  print(f"New Pairs Generated    : {len(newly_generated_pairs):,}")
  print(f"Duplicate Pairs Skipped: {duplicates_skipped:,}")
  print(f"Total Unique Cumulative: {len(hf_dataset):,}")
  print(f"Training Set (80%)     : {len(train_data):,}")
  print(f"Evaluation Set (20%)   : {len(eval_data):,}")
  print(f"Raw Appended File      : {RAW_JSONL_PATH}")
  print(f"Final Train/Eval Paths : {train_path} | {eval_path}")
  print("===================================================================")

# Part B2 — QLoRA Fine-Tuning with One Adapter Configurations

In [ ]:
import os
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from trl import SFTTrainer, SFTConfig


# ==============================================================================
# 1. PATHS
# ==============================================================================

BASE_MODEL_PATH = "./SmolLM2_CPT_Project/cpt_smollm2_ubuntu"

JSON_DATASET_PATH = (
    "./SmolLM2_CPT_Project/all_generated_pairs.jsonl"
)

OUTPUT_DIR = "./qlora_adapter_v2"

HF_TOKENIZER = "HuggingFaceTB/SmolLM2-360M"


# ==============================================================================
# 2. QLORA PARAMETERS
# ==============================================================================

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]

LEARNING_RATE = 1e-4
NUM_EPOCHS = 2


print("============================================================")
print("QLoRA CONFIG")
print("============================================================")
print(f"LoRA rank       : {LORA_R}")
print(f"LoRA alpha      : {LORA_ALPHA}")
print(f"LoRA dropout    : {LORA_DROPOUT}")
print(f"Target modules  : {TARGET_MODULES}")
print(f"Learning rate   : {LEARNING_RATE}")
print(f"Epochs          : {NUM_EPOCHS}")
print("============================================================")


# ==============================================================================
# 3. LOAD DATASET
# ==============================================================================

print("\nLoading dataset...")

dataset = Dataset.from_json(JSON_DATASET_PATH)

print(f"Total examples: {len(dataset):,}")


# ==============================================================================
# 4. BASIC DATA VALIDATION
# ==============================================================================

def is_valid(example):

    instruction = str(example["instruction"]).strip()
    response = str(example["response"]).strip()

    if not instruction or not response:
        return False

    bad_patterns = [
        "meticulassistant",
        "of 1139",
        "<!--",
        "<rule>",
        "</rule>",
    ]

    combined = (instruction + " " + response).lower()

    for pattern in bad_patterns:
        if pattern.lower() in combined:
            return False

    return True


dataset = dataset.filter(is_valid)

print(f"After basic filtering: {len(dataset):,}")


# ==============================================================================
# 5. LOAD TOKENIZER
# ==============================================================================

print("\nLoading tokenizer from Hugging Face:")
print(HF_TOKENIZER)

tokenizer = AutoTokenizer.from_pretrained(
    HF_TOKENIZER,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ==============================================================================
# 6. VERIFY SPECIAL TOKENS
# ==============================================================================

print("\nSpecial token check:")

print(
    "<|im_start|> token ID:",
    tokenizer.convert_tokens_to_ids("<|im_start|>")
)

print(
    "<|im_end|> token ID:",
    tokenizer.convert_tokens_to_ids("<|im_end|>")
)


# ==============================================================================
# 7. FORMAT DATASET
# ==============================================================================

def format_example(example):

    instruction = str(example["instruction"]).strip()
    response = str(example["response"]).strip()

    formatted_text = (
        f"<|im_start|>user\n"
        f"{instruction}"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"{response}"
        f"<|im_end|>\n"
    )

    return {
        "text": formatted_text
    }


print("\nApplying SmolLM2 conversation format...")

dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names,
)


# ==============================================================================
# 8. CHECK FORMATTED EXAMPLE
# ==============================================================================

print("\n============================================================")
print("SAMPLE TRAINING EXAMPLE")
print("============================================================")

print(dataset[0]["text"])

print("============================================================")


# ==============================================================================
# 9. TRAIN / EVAL SPLIT
# ==============================================================================

split_dataset = dataset.train_test_split(
    test_size=0.20,
    seed=42,
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("\n============================================================")
print("DATASET SPLIT")
print("============================================================")
print(f"Total : {len(dataset):,}")
print(f"Train : {len(train_dataset):,}")
print(f"Eval  : {len(eval_dataset):,}")
print("============================================================")


# ==============================================================================
# 10. 4-BIT QUANTIZATION
# ==============================================================================

# Disable BF16.
# Use FP16 for the 4-bit computation dtype.

use_bf16 = False
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)


# ==============================================================================
# 11. LOAD CPT MODEL
# ==============================================================================

print("\nLoading CPT model from:")
print(BASE_MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False


# ==============================================================================
# 12. PREPARE MODEL FOR QLORA
# ==============================================================================

model = prepare_model_for_kbit_training(model)


# ==============================================================================
# 13. ATTACH LORA
# ==============================================================================

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(
    model,
    peft_config,
)

model.print_trainable_parameters()


# ==============================================================================
# 14. TRAINING CONFIG
# ==============================================================================

training_args = SFTConfig(

    output_dir=OUTPUT_DIR,

    # Maximum sequence length
    max_length=1024,

    # Batch
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    # Effective batch size:
    # 2 * 4 = 8
    gradient_accumulation_steps=4,

    # Optimization
    learning_rate=LEARNING_RATE,

    num_train_epochs=NUM_EPOCHS,

    lr_scheduler_type="cosine",

    optim="paged_adamw_8bit",

    weight_decay=0.01,

    max_grad_norm=1.0,

    # Precision
    # AMP disabled to avoid the BF16 GradScaler error.
    bf16=False,
    fp16=False,

    # Evaluation
    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    # Logging
    logging_steps=10,

    report_to="none",

    seed=42,
)


# ==============================================================================
# 15. TRAINER
# ==============================================================================

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    args=training_args,
)


# ==============================================================================
# 16. TRAIN
# ==============================================================================

print("\n")
print("============================================================")
print("STARTING QLORA TRAINING")
print("============================================================")
print(f"Training examples : {len(train_dataset):,}")
print(f"Evaluation examples: {len(eval_dataset):,}")
print(f"Learning rate     : {LEARNING_RATE}")
print(f"Epochs            : {NUM_EPOCHS}")
print(f"LoRA rank         : {LORA_R}")
print(f"Target modules    : {TARGET_MODULES}")
print("============================================================")
print()

trainer.train()


# ==============================================================================
# 16.1 LOSS CURVE
# ==============================================================================

log_history = trainer.state.log_history

train_steps = []
train_losses = []

eval_steps = []
eval_losses = []

for log in log_history:

    if "loss" in log and "eval_loss" not in log:
        train_steps.append(log["step"])
        train_losses.append(log["loss"])

    if "eval_loss" in log:
        eval_steps.append(log["step"])
        eval_losses.append(log["eval_loss"])


plt.figure(figsize=(10, 6))

if train_losses:
    plt.plot(
        train_steps,
        train_losses,
        label="Training Loss",
    )

if eval_losses:
    plt.plot(
        eval_steps,
        eval_losses,
        marker="o",
        label="Evaluation Loss",
    )

plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("QLoRA Training Loss Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()

loss_curve_path = os.path.join(
    OUTPUT_DIR,
    "loss_curve.png",
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.savefig(
    loss_curve_path,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print(f"\nLoss curve saved to:")
print(loss_curve_path)


# ==============================================================================
# 17. SAVE FINAL ADAPTER
# ==============================================================================

final_adapter_path = os.path.join(
    OUTPUT_DIR,
    "final_adapter",
)

trainer.model.save_pretrained(
    final_adapter_path
)

tokenizer.save_pretrained(
    final_adapter_path
)

print("\n============================================================")
print("TRAINING COMPLETE")
print("============================================================")
print(f"Adapter saved to:")
print(final_adapter_path)
print(f"Loss curve saved to:")
print(loss_curve_path)
print("============================================================")

# Part B3 — Evaluation Analysis

In [ ]:
import torch
import textwrap
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel


# ==============================================================================
# 1. PATHS
# ==============================================================================

BASE_MODEL_PATH = (
    "./SmolLM2_CPT_Project/cpt_smollm2_ubuntu"
)

ADAPTER_PATH = "./qlora_adapter_v2/final_adapter"

# IMPORTANT:
# Same tokenizer as the original CPT base model.
HF_TOKENIZER = "HuggingFaceTB/SmolLM2-360M"


# ==============================================================================
# 2. TEST PROMPTS
# ==============================================================================

domain_prompts = [
    "How do I configure a static IP address in Ubuntu Server using Netplan?",
    "Explain how to manage system services using systemctl in Ubuntu.",
    "What steps are required to set up Uncomplicated Firewall (UFW) on Ubuntu?"
]


# ==============================================================================
# 3. LOAD TOKENIZER FROM HUGGING FACE
# ==============================================================================

print(f"Loading tokenizer from: {HF_TOKENIZER}")

tokenizer = AutoTokenizer.from_pretrained(
    HF_TOKENIZER,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ==============================================================================
# 4. VERIFY SPECIAL TOKENS
# ==============================================================================

im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

print("\nSpecial tokens:")
print(f"<|im_start|> ID: {im_start_id}")
print(f"<|im_end|>   ID: {im_end_id}")
print(f"EOS token ID : {tokenizer.eos_token_id}")
print(f"PAD token ID : {tokenizer.pad_token_id}")


if im_start_id == tokenizer.unk_token_id:
    raise ValueError(
        "<|im_start|> is not present in the tokenizer."
    )

if im_end_id == tokenizer.unk_token_id:
    raise ValueError(
        "<|im_end|> is not present in the tokenizer."
    )


# ==============================================================================
# 5. 4-BIT QUANTIZATION
# ==============================================================================

use_bf16 = False

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=(
        torch.bfloat16 if use_bf16 else torch.float16
    ),
    bnb_4bit_use_double_quant=True,
)


# ==============================================================================
# 6. LOAD CPT BASE MODEL
# ==============================================================================

print("\n============================================================")
print("Loading CPT BASE model")
print("============================================================")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("CPT model loaded.")


# ==============================================================================
# 7. CHECK VOCABULARY COMPATIBILITY
# ==============================================================================

model_vocab_size = base_model.get_input_embeddings().weight.shape[0]
tokenizer_vocab_size = len(tokenizer)

print("\nVocabulary:")
print(f"Model tokenizer embeddings : {model_vocab_size}")
print(f"Tokenizer length            : {tokenizer_vocab_size}")


if model_vocab_size != tokenizer_vocab_size:
    raise ValueError(
        f"Vocabulary mismatch!\n"
        f"Model embeddings: {model_vocab_size}\n"
        f"Tokenizer: {tokenizer_vocab_size}\n"
        f"Do NOT resize automatically. "
        f"Use the tokenizer that matches your original CPT model."
    )


# ==============================================================================
# 8. LOAD QLORA ADAPTER
# ==============================================================================

print("\n============================================================")
print("Loading QLoRA adapter")
print("============================================================")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

print("QLoRA adapter loaded successfully.")

model.eval()


# ==============================================================================
# 9. DEVICE
# ==============================================================================

device = next(model.parameters()).device

print(f"Model device: {device}")


# ==============================================================================
# 10. GENERATION
# ==============================================================================

print("\n")
print("================ QUALITATIVE EVALUATION ====================")


for i, prompt in enumerate(domain_prompts, 1):

    # EXACT SAME FORMAT USED DURING TRAINING
    formatted_prompt = (
        "<|im_start|>user\n"
        f"{prompt}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():

        output = model.generate(
            **inputs,

            max_new_tokens=150,

            do_sample=False,

            repetition_penalty=1.05,

            pad_token_id=tokenizer.pad_token_id,

            eos_token_id=im_end_id,
        )


    # Only decode newly generated tokens
    generated_tokens = output[0][input_length:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()


    print(f"\n[Prompt {i}]")
    print(textwrap.fill(prompt, width=100))

    print("\n[QLoRA Completion]")
    print(textwrap.fill(generated_text, width=100))

    print("-" * 60)

# Model Comparison: Base vs CPT vs CPT + QLoRA

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel
import textwrap


# ============================================================
# PATHS
# ============================================================

HF_BASE_MODEL = "HuggingFaceTB/SmolLM2-360M"

CPT_MODEL_PATH = (
    "./SmolLM2_CPT_Project/"
    "cpt_smollm2_ubuntu"
)

QLORA_ADAPTER_PATH = "/content/qlora_adapter_v2/final_adapter"


# ============================================================
# EVALUATION PROMPTS
# ============================================================

domain_prompts = [

    "How do I configure a static IP address in Ubuntu Server using Netplan?",

    "Explain how to manage system services using systemctl in Ubuntu.",

    "What steps are required to set up Uncomplicated Firewall (UFW) on Ubuntu?",
]


# ============================================================
# TOKENIZER
# ============================================================

print("=" * 70)
print("Loading tokenizer")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    HF_BASE_MODEL,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

print("Tokenizer:", HF_BASE_MODEL)
print("<|im_start|> ID:", im_start_id)
print("<|im_end|>   ID:", im_end_id)
print("EOS token ID :", tokenizer.eos_token_id)
print("PAD token ID :", tokenizer.pad_token_id)


# ============================================================
# 4-BIT CONFIG
# ============================================================

use_bf16 = False

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)


# ============================================================
# HELPER: LOAD MODEL
# ============================================================

def load_model(model_path, adapter_path=None):

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model_vocab_size = model.get_input_embeddings().weight.shape[0]
    tokenizer_vocab_size = len(tokenizer)

    if model_vocab_size != tokenizer_vocab_size:
        raise ValueError(
            f"Vocabulary mismatch!\n"
            f"Model: {model_vocab_size}\n"
            f"Tokenizer: {tokenizer_vocab_size}"
        )

    if adapter_path is not None:

        model = PeftModel.from_pretrained(
            model,
            adapter_path,
        )

    model.eval()

    return model


# ============================================================
# LOAD BASE MODEL
# ============================================================

print("\n" + "=" * 70)
print("1. LOADING ORIGINAL BASE MODEL")
print("=" * 70)

base_model = load_model(HF_BASE_MODEL)

print("Base model loaded successfully.")


# ============================================================
# LOAD CPT MODEL
# ============================================================

print("\n" + "=" * 70)
print("2. LOADING CPT MODEL")
print("=" * 70)

cpt_model = load_model(CPT_MODEL_PATH)

print("CPT model loaded successfully.")


# ============================================================
# LOAD CPT + QLoRA
# ============================================================

print("\n" + "=" * 70)
print("3. LOADING CPT + QLoRA")
print("=" * 70)

qlora_model = load_model(
    CPT_MODEL_PATH,
    QLORA_ADAPTER_PATH,
)

print("CPT + QLoRA loaded successfully.")


# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate(model, prompt):

    formatted_prompt = (
        "<|im_start|>user\n"
        f"{prompt}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
    )

    device = next(model.parameters()).device

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():

        output = model.generate(
            **inputs,

            max_new_tokens=150,

            do_sample=False,

            repetition_penalty=1.05,

            pad_token_id=tokenizer.pad_token_id,

            eos_token_id=im_end_id,
        )

    generated_tokens = output[0][input_length:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    return generated_text


# ============================================================
# RUN COMPARISON
# ============================================================

print("\n")
print("#" * 90)
print("#" + " MODEL COMPARISON ".center(88) + "#")
print("#" * 90)


for i, prompt in enumerate(domain_prompts, start=1):

    print("\n\n")
    print("█" * 90)
    print(f"█  PROMPT {i}")
    print("█" * 90)

    print()
    print(textwrap.fill(
        prompt,
        width=88,
        initial_indent="   ",
        subsequent_indent="   "
    ))


    # --------------------------------------------------------
    # BASE
    # --------------------------------------------------------

    print("\n" + "─" * 90)
    print("  BASE — SmolLM2-360M")
    print("─" * 90)

    base_output = generate(
        base_model,
        prompt,
    )

    print()
    print(textwrap.fill(
        base_output,
        width=88,
        initial_indent="   ",
        subsequent_indent="   "
    ))


    # --------------------------------------------------------
    # CPT
    # --------------------------------------------------------

    print("\n" + "─" * 90)
    print("  CPT — Ubuntu Continued Pretraining")
    print("─" * 90)

    cpt_output = generate(
        cpt_model,
        prompt,
    )

    print()
    print(textwrap.fill(
        cpt_output,
        width=88,
        initial_indent="   ",
        subsequent_indent="   "
    ))


    # --------------------------------------------------------
    # QLoRA
    # --------------------------------------------------------

    print("\n" + "─" * 90)
    print("  CPT + QLoRA — SFT")
    print("─" * 90)

    qlora_output = generate(
        qlora_model,
        prompt,
    )

    print()
    print(textwrap.fill(
        qlora_output,
        width=88,
        initial_indent="   ",
        subsequent_indent="   "
    ))


    print("\n" + "█" * 90)


# ============================================================
# CLEANUP
# ============================================================

del base_model
del cpt_model
del qlora_model

torch.cuda.empty_cache()

print("\n" + "=" * 90)
print("DONE")
print("=" * 90)

### **Qualitative Evaluation Analysis & Key Insights**

#### **1. Base Model vs. CPT Model (Knowledge Enhancement)**
* **Base Model**: Lacks specific Ubuntu/Linux domain depth. Often generates generic, inaccurate, or unrelated responses (such as mixing up Linux configurations with Windows UI or missing technical specifics like Netplan/BIND9 syntax).
* **CPT (Continual Pre-trained) Model**: Exhibits enhanced familiarity with the Ubuntu-specific technical terms, tools (Netplan, BIND9, UFW), and configuration syntax. However, as expected from raw pre-training, it struggles with the **instruction-following structure** and conversational flow, frequently returning incomplete configurations or continuing to predict subsequent raw text rather than a direct answer.

#### **2. CPT vs. QLoRA SFT Model (Instruction Alignment)**
* **QLoRA Fine-Tuning**: Successfully bridges the gap between raw domain knowledge and practical usability.
* **Formatting & Control**: The model strictly adheres to the custom ChatML style markup (`<|im_start|>`/`<|im_end|>`), stopping cleanly on target tokens without hallucinating metadata or trailing training artifacts.
* **Technical Accuracy**: Delivers clear, step-by-step procedures (e.g., systemctl commands, UFW firewall configurations) and accurate configuration templates formatted for immediate developer use.